# 🎙️ GHX VOICE STUDIO V5

Multilingual voice cloning, voice design and batch production in one workspace.

**Requirement:** Runtime → Change runtime type → **T4 GPU**. Complete Step 1 access verification, then Step 2 setup and Step 3 launch.

### 1. Verify Cloudflare access
Enter the registered customer email. Colab will verify the signed-in Google account, then check the GHX Cloudflare license server. The studio will not launch unless access is active.

In [ ]:
#@title 🔐 Step 1 — Verify GHX Access { display-mode: "form" }
EMAIL = '' #@param {type:"string"}

import json, urllib.request, urllib.error
from google.colab import auth
import google.auth
from google.auth.transport.requests import Request

CLOUDFLARE_ACCESS_URL = 'https://ghx-voice-studio-server.autoflow-ghx.workers.dev/api/access'
email = EMAIL.strip().lower()
if not email:
    raise RuntimeError('Please enter your registered Gmail in the EMAIL field, then run this cell again.')

print('Connecting to Google account verification...')
auth.authenticate_user()
creds, _ = google.auth.default()
creds.refresh(Request())
req = urllib.request.Request(
    'https://www.googleapis.com/oauth2/v2/userinfo',
    headers={'Authorization': f'Bearer {creds.token}', 'Accept': 'application/json'}
)
with urllib.request.urlopen(req, timeout=20) as response:
    google_email = json.loads(response.read().decode()).get('email', '').strip().lower()

if not google_email:
    raise RuntimeError('Could not verify the signed-in Google account. Please reconnect Google in Colab.')
print(f'Google account: {google_email}')
if google_email != email:
    raise RuntimeError(f'Gmail mismatch. Colab is signed in as {google_email}, but the entered email is {email}. Sign in with the registered Gmail.')

payload = json.dumps({'email': email}).encode('utf-8')
request = urllib.request.Request(
    CLOUDFLARE_ACCESS_URL, data=payload,
    headers={'Content-Type': 'application/json', 'Accept': 'application/json'},
    method='POST'
)
try:
    with urllib.request.urlopen(request, timeout=20) as response:
        result = json.loads(response.read().decode())
except urllib.error.HTTPError as exc:
    detail = exc.read().decode(errors='replace')[:300]
    raise RuntimeError(f'Cloudflare access server returned HTTP {exc.code}: {detail}')
except Exception as exc:
    raise RuntimeError(f'Could not reach the GHX Cloudflare access server: {type(exc).__name__}: {exc}')

status = result.get('status', '')
if status == 'ok':
    print('✅ ACCESS GRANTED')
    if result.get('plan'): print('Plan:', result['plan'])
    if result.get('expires'): print('Valid through:', result['expires'])
    print('➡️ Run Step 2 — Setup.')
elif status == 'expired':
    raise RuntimeError(f'⏳ Subscription expired. Expiry: {result.get("expires") or "not provided"}. Contact GHX Studio for renewal.')
elif status == 'off':
    raise RuntimeError('🔴 GHX Voice Studio is temporarily disabled by the owner. Please try again later.')
else:
    raise RuntimeError('🔒 ACCESS DENIED. This Gmail is not active in the GHX Cloudflare client list. Contact GHX Studio.')


### 2. Setup
Downloads the V5 application and installs its dependencies.

In [ ]:
!rm -rf /content/ghx_voice_studio_v5 && git clone -q --depth 1 https://github.com/GHX-STUDIO-786/GHX-VOICE-STUDIO-V5.git /content/ghx_voice_studio_v5
%cd -q /content/ghx_voice_studio_v5
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], capture_output=True, text=True)
print("Setup complete. Run Step 3 — Launch." if r.returncode == 0 else "Setup failed:\n" + r.stderr[-800:])

### 3. Launch
Starts the studio. Open the public link shown below and keep this cell running while you work.

In [ ]:
%cd -q /content/ghx_voice_studio_v5
%run app.py